In [5]:
import numpy as np
import pandas as pd

array([1, 2, 3])

In [8]:
# read hourly_normalized from Snowflake-Labs/shavedice-dataset
df = pd.read_parquet(
    "https://github.com/Snowflake-Labs/shavedice-dataset/raw/main/hourly_normalized.parquet"
)

# print the first 5 rows of the dataframe
df.head()

,USAGE_HOUR,REGION_NUM,INSTANCE_TYPE,NORM_USAGE
0,2021-02-01 00:00:00+00:00,2,A,172.0
1,2021-02-01 00:00:00+00:00,4,F,1.0
2,2021-02-01 00:00:00+00:00,1,I,9.0
3,2021-02-01 00:00:00+00:00,4,I,8.0
4,2021-02-01 00:00:00+00:00,3,I,6.0


In [9]:
df.describe()

,NORM_USAGE
count,524832.000000
mean,87.626782
std,174.811591
min,1.000000
25%,4.000000
50%,7.000000
75%,18.000000
max,1000.000000


In [10]:
from scipy.optimize import minimize_scalar
import numpy as np


def total_cost(demands, commitment, on_demand_premium=2.1):
    """Calculate total cost for a given commitment level."""
    covered = np.minimum(demands, commitment)  # SP covers up to commitment
    unused = commitment - covered  # wasted commitment
    excess = np.maximum(demands - commitment, 0)  # on-demand portion

    return covered.sum() + unused.sum() + (excess * on_demand_premium).sum()


# Find optimal commitment
demands = df["NORM_USAGE"].values
result = minimize_scalar(
    lambda c: total_cost(demands, c),
    bounds=(demands.min(), demands.max()),
    method="bounded",
)

print(f"Optimal commitment: {result.x:.2f}")
print(f"Total cost: {result.fun:.2f}")
print(f"Evaluations: {result.nfev}")  # typically 10-15

ModuleNotFoundError: No module named 'scipy'